In [8]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import ast
import folium

In [5]:
#Load Data
ACLED_20_23_all = pd.read_csv('../Spatial Joins/ACLED Join/US2020_2023.csv')

#Clean Data, filtering for PA and 2023!
ACLED_21_PA = ACLED_20_23_all[(ACLED_20_23_all['admin1'] == 'Pennsylvania') & (ACLED_20_23_all['year'] == 2021)]

#Transform ACLED Data into GeoDataframe
#iterating over each row of the ACLED dataset to create list of point objects using lat and lon values
geometry = [Point(xy) for xy in zip(ACLED_21_PA['longitude'], ACLED_21_PA['latitude'])]

#creating new Geo dataframe using the Geodatapoints extracted in the previous step
gdf_ACLED_21_PA = gpd.GeoDataFrame(ACLED_21_PA, geometry=geometry)

### Parsing Vector Layer (ZIP Code Polygons)

In [6]:
#Parsing Geojson file 
gdf_ZIP_PA = gpd.read_file('../Spatial Joins/ACLED Join/pa_pennsylvania_zip_codes_geo.min.json')

### Overlaying Maps

In [9]:
#Defining center of map using bounds of the ZIP Code polygons --> calculating by averaging lat and long values to find midpoint
bounds = gdf_ZIP_PA.total_bounds
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

#Initialising map centrered around the calculated center 
map_PA = folium.Map(location=center, zoom_start=10)

#Adding ZIP Code polygon layer to initialised map 
folium.GeoJson(gdf_ZIP_PA, name="ZIP Codes PA").add_to(map_PA)

#Adding ACLED data points to initialised map 
for idx, row in ACLED_21_PA.iterrows():
    folium.Marker([row['latitude'], row['longitude']]).add_to(map_PA)

### Spatial Join 

In [10]:
spatial_join_PA_21 = gpd.sjoin(gdf_ACLED_21_PA, gdf_ZIP_PA, how="inner", op='within')

/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3493: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):
/var/folders/4s/d5gptfss61l9fqz7rk3r5m880000gn/T/ipykernel_85821/1419187888.py:1: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  spatial_join_PA_21 = gpd.sjoin(gdf_ACLED_21_PA, gdf_ZIP_PA, how="inner", op='within')


In [11]:
#Store Data in GeoJSON file 
spatial_join_PA_21.to_file("joined_ACLED_ZIP_21_PA.geojson", driver='GeoJSON')

#Transforming Geojson file into GeoDataframe
geojson_path_21 = 'joined_ACLED_ZIP_21_PA.geojson'
gdf_PA_21 = gpd.read_file(geojson_path_21)

#Converting Geometries to WKT
gdf_PA_21['geometry'] = gdf_PA_21['geometry'].apply(lambda x: x.wkt)

#Convert to csv
#Saving this as csv is unnecessary, will probably delete this later
gdf_PA_21.to_csv("joined_ACLED_ZIP_21_PA.csv", index=False)

/var/folders/4s/d5gptfss61l9fqz7rk3r5m880000gn/T/ipykernel_85821/3880325397.py:9: UserWarning: Geometry column does not contain geometry.
  gdf_PA_21['geometry'] = gdf_PA_21['geometry'].apply(lambda x: x.wkt)


### Restructuring data to extract conflict events per month

In [13]:
#Loading data
ACLED_21_df = pd.read_csv('../feature space/joined_ACLED_ZIP_21_PA.csv')

In [14]:
#Extracting year and month from date column

#Parsing date column to datetime
ACLED_21_df['parsed_date'] = pd.to_datetime(ACLED_21_df['event_date'], format='%d %B %Y')

#Extracting year and month
ACLED_21_df['event_MY'] = ACLED_21_df['parsed_date'].dt.to_period('M')

In [28]:
#Aggregation function for nr of events

#Defining aggregation across features
aggregation_function_21 = {
    'admin2': 'first',  
}

#### Grouping by ZIP Code and Month

In [29]:
#Grouping by ZIP Code and Month
#agg aggregates over the previously specified parameters
grouped_21 = ACLED_21_df.groupby(['ZCTA5CE10', 'event_MY']).agg(aggregation_function_21).reset_index()

#Adding event count per ZIP Code and Month
grouped_21['event_count'] = ACLED_21_df.groupby(['ZCTA5CE10', 'event_MY']).size().reset_index(name='event_count')['event_count']

print(grouped_21.head(10))

   ZCTA5CE10 event_MY     admin2  event_count
0      15009  2021-01     Beaver            1
1      15009  2021-03     Beaver            1
2      15009  2021-05     Beaver            1
3      15014  2021-03  Allegheny            1
4      15014  2021-04  Allegheny            1
5      15017  2021-01  Allegheny            1
6      15017  2021-02  Allegheny            1
7      15017  2021-09  Allegheny            1
8      15024  2021-03  Allegheny            1
9      15044  2021-03  Allegheny            1


In [30]:
#Date range for 2021

date_range_21 = pd.period_range(start='2021-01', end='2021-12', freq='M')
print(date_range_21)

PeriodIndex(['2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06',
             '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12'],
            dtype='period[M]')


In [31]:
#Dataframe mapping ZIP Codes to Counties 
zip_county_21 = ACLED_21_df[['ZCTA5CE10', 'admin2']].drop_duplicates()

#New dataframe combining date range and ZIP Code
all_combinations_21 = pd.MultiIndex.from_product([ACLED_21_df['ZCTA5CE10'].unique(), date_range_21], names=['ZCTA5CE10', 'event_MY'])

#Initializing df with all combinations 
restructured_df_21 = pd.DataFrame(index=all_combinations_21).reset_index()

In [32]:
#Merge 1
#Merging zip_county df with initialized df for all combinations
restructured_df_21 = restructured_df_21.merge(zip_county_21, on='ZCTA5CE10', how='left')

In [33]:
#Merge 2 
#Merging initialized df with grouped data holding aggregates and counts
restructured_df_21 = restructured_df_21.merge(grouped_21, on=['ZCTA5CE10', 'admin2', 'event_MY'], how='left')

In [34]:
#Filling missing values + converting to int
restructured_df_21['event_count'] = restructured_df_21['event_count'].fillna(0).astype(int)

In [35]:
#Sorting df by ZIP and time
restructured_df_21 = restructured_df_21.sort_values(by=['ZCTA5CE10', 'event_MY'])

#Renaming columns
restructured_df_21.rename(columns={'ZCTA5CE10': 'ZCTA'}, inplace=True)

In [36]:
#Exporting to csv
restructured_df_21.to_csv("ACLED_Events_21_Clean.csv", index=False)